# Optuna Search

## A. Setup

In [2]:
import gc
import json
import random
import tempfile
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import optuna
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

SEED = 42
MAX_EPOCHS_PER_PHASE = 50
EARLY_STOPPING_PATIENCE = 5
IMAGE_SIZE = (224, 224)
HIDDEN_DIM = 512
NUM_CLASSES = 4
FINE_TUNE_LR_DIVISOR = 10
OPTIMIZERS = ("Adam", "AdamW")
LEARNING_RATES = (1e-3, 1e-4, 1e-5)
DROPOUT_RATES = (0.4, 0.5)
FINETUNE_LEVELS = ("frozen", "medium", "deep")
CLASS_NAMES = ("leaf_curl", "leaf_spot", "yellowish", "healthy_leaf")
CLASS_DISPLAY_NAMES = ("Leaf Curl", "Leaf Spot", "Yellowish", "Healthy Leaf")
DATA_ROOT = Path("data/processed")
CHECKPOINT_DIR = Path("checkpoints")
REPORT_DIR = Path("reports")
SCREENING_PATH = REPORT_DIR / "screening_results.csv"
HISTORY_DIR = REPORT_DIR / "histories"
CONFUSION_DIR = REPORT_DIR / "confusion"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## B. Data

In [3]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1), fill=0),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
VALIDATION_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

def set_class_order(dataset):
    old_indices = {name: index for index, name in enumerate(dataset.classes)}
    old_to_new = {old_indices[name]: index for index, name in enumerate(CLASS_NAMES)}
    dataset.classes = list(CLASS_NAMES)
    dataset.class_to_idx = {name: index for index, name in enumerate(CLASS_NAMES)}
    dataset.samples = [(path, old_to_new[target]) for path, target in dataset.samples]
    dataset.imgs = dataset.samples
    dataset.targets = [target for _, target in dataset.samples]
    return dataset

train_dataset = set_class_order(datasets.ImageFolder(DATA_ROOT / "train", transform=TRAIN_TRANSFORM))
validation_dataset = set_class_order(datasets.ImageFolder(DATA_ROOT / "validation", transform=VALIDATION_TRANSFORM))
assert train_dataset.classes == validation_dataset.classes == list(CLASS_NAMES)

## C. Core code

In [4]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
    generator=torch.Generator(); generator.manual_seed(seed)
    return generator

def build_dataloaders(batch_size,seed):
    generator=seed_everything(seed)
    return (DataLoader(train_dataset,batch_size=batch_size,shuffle=True,generator=generator,num_workers=0,pin_memory=True),
            DataLoader(validation_dataset,batch_size=batch_size,shuffle=False,num_workers=0,pin_memory=True))

def build_head(input_features,dropout):
    return nn.Sequential(nn.Dropout(dropout),nn.Linear(input_features,HIDDEN_DIM),nn.ReLU(),nn.Dropout(dropout),nn.Linear(HIDDEN_DIM,NUM_CLASSES))

class FeatureBackbone(nn.Module):
    def __init__(self,features):
        super().__init__(); self.features=features; self.pool=nn.AdaptiveAvgPool2d(1)
    def forward(self,images):
        return self.pool(self.features(images)).flatten(1)

class SingleBackboneModel(nn.Module):
    def __init__(self,features,dropout):
        super().__init__(); self.features=features; self.pool=nn.AdaptiveAvgPool2d(1); self.head=build_head(1280,dropout)
    def forward(self,images):
        return self.head(self.pool(self.features(images)).flatten(1))

class FusionModel(nn.Module):
    def __init__(self,mobile_features,efficient_features,dropout):
        super().__init__(); self.mobilenet=FeatureBackbone(mobile_features); self.efficientnet=FeatureBackbone(efficient_features); self.head=build_head(2560,dropout)
    def forward(self,images):
        return self.head(torch.cat((self.mobilenet(images),self.efficientnet(images)),dim=1))

def build_base_model(model_name,dropout):
    if model_name=="mobilenet": backbone=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    elif model_name=="efficientnet": backbone=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    else: raise ValueError(f"Unsupported model: {model_name}")
    return SingleBackboneModel(backbone.features,dropout).to(DEVICE)

def build_hybrid_model(dropout):
    mobile=models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    efficient=models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    return FusionModel(mobile.features,efficient.features,dropout).to(DEVICE)

def apply_finetune_level(model,finetune_level):
    if not isinstance(model,FusionModel):
        for parameter in model.parameters(): parameter.requires_grad=True
        return
    if finetune_level not in FINETUNE_LEVELS: raise ValueError("Hybrid scenarios require a fine-tuning level.")
    for parameter in model.parameters(): parameter.requires_grad=False
    for parameter in model.head.parameters(): parameter.requires_grad=True
    if finetune_level=="frozen": return
    ranges={"medium":((14,19),(6,9)),"deep":((10,19),(4,9))}
    for backbone,(start,stop) in zip((model.mobilenet,model.efficientnet),ranges[finetune_level]):
        for block in backbone.features[start:stop]:
            for parameter in block.parameters(): parameter.requires_grad=True

def set_training_mode(model,phase):
    model.train()
    if isinstance(model,FusionModel):
        if phase==1: model.mobilenet.eval(); model.efficientnet.eval()
        for module in model.modules():
            if isinstance(module,nn.modules.batchnorm._BatchNorm): module.eval()
    return model

def build_optimizer(model,optimizer_name,learning_rate):
    parameters=[p for p in model.parameters() if p.requires_grad]
    if optimizer_name=="Adam": return torch.optim.Adam(parameters,lr=learning_rate,weight_decay=0.0)
    if optimizer_name=="AdamW": return torch.optim.AdamW(parameters,lr=learning_rate,weight_decay=0.01)
    raise ValueError(f"Unsupported optimizer: {optimizer_name}")

def evaluate_model(model,loader):
    model.eval(); criterion=nn.CrossEntropyLoss(); loss_sum=0.0; targets=[]; predictions=[]; probabilities=[]
    with torch.no_grad():
        for images,labels in loader:
            images=images.to(DEVICE,non_blocking=True); labels_device=labels.to(DEVICE,non_blocking=True)
            logits=model(images); loss_sum+=criterion(logits,labels_device).item()*len(labels)
            targets.extend(labels.numpy().tolist()); predictions.extend(logits.argmax(1).cpu().numpy().tolist())
            probabilities.append(torch.softmax(logits,dim=1).cpu().numpy())
    if not targets: raise ValueError("Evaluation data loader is empty.")
    return np.asarray(targets),np.asarray(predictions),np.concatenate(probabilities),loss_sum/len(targets)

def compute_metrics(targets,predictions,probabilities,loss):
    return {"Accuracy":accuracy_score(targets,predictions),
        "Precision":precision_score(targets,predictions,average="macro",zero_division=0),
        "Recall":recall_score(targets,predictions,average="macro",zero_division=0),
        "F1":f1_score(targets,predictions,average="macro",zero_division=0),"Loss":float(loss),
        "AUC":roc_auc_score(targets,probabilities,labels=np.arange(NUM_CLASSES),average="macro",multi_class="ovr")}

def measure_efficiency(model):
    params=sum(p.numel() for p in model.parameters())
    with tempfile.NamedTemporaryFile(suffix=".pt",delete=False) as file: path=Path(file.name)
    try: torch.save(model.state_dict(),path); size=path.stat().st_size/(1024**2)
    finally: path.unlink(missing_ok=True)
    model.eval(); sample=torch.zeros(1,3,*IMAGE_SIZE,device=DEVICE); timings=[]
    with torch.no_grad():
        for _ in range(20): model(sample)
        if DEVICE.type=="cuda": torch.cuda.synchronize()
        for _ in range(100):
            start=time.perf_counter(); model(sample)
            if DEVICE.type=="cuda": torch.cuda.synchronize()
            timings.append((time.perf_counter()-start)*1000)
    return {"Params":params,"Size (MB)":size,"Latency (ms/img)":float(np.mean(timings))}

def train_one_phase(model,train_loader,validation_loader,optimizer,scaler,phase,show_epoch_log):
    criterion=nn.CrossEntropyLoss(); best_loss=float("inf"); best_state=None; best_epoch=0; wait=0; history=[]
    for epoch in range(1,MAX_EPOCHS_PER_PHASE+1):
        set_training_mode(model,phase); started=time.perf_counter(); loss_sum=0.0; correct=0; count=0
        for images,targets in train_loader:
            images=images.to(DEVICE,non_blocking=True); targets=targets.to(DEVICE,non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=="cuda"):
                logits=model(images); loss=criterion(logits,targets)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum+=loss.item()*len(targets); correct+=(logits.argmax(1)==targets).sum().item(); count+=len(targets)
        if not count: raise ValueError("Training data loader is empty.")
        train_loss=loss_sum/count; train_accuracy=correct/count
        targets_all,predictions,_,val_loss=evaluate_model(model,validation_loader); val_accuracy=accuracy_score(targets_all,predictions)
        seconds=time.perf_counter()-started
        history.append({"phase":phase,"epoch":epoch,"train_loss":train_loss,"train_accuracy":train_accuracy,"validation_loss":val_loss,"validation_accuracy":val_accuracy,"epoch_seconds":seconds})
        if show_epoch_log: print(f"phase {phase} | epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_accuracy:.4f} | val loss {val_loss:.4f} acc {val_accuracy:.4f} | {seconds:.1f} s")
        if val_loss<best_loss:
            best_loss=val_loss; best_epoch=epoch; wait=0
            best_state={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
        else: wait+=1
        if wait>=EARLY_STOPPING_PATIENCE: break
    if best_state is None: raise RuntimeError("Training did not produce a best state.")
    model.load_state_dict(best_state)
    if show_epoch_log: print(f"phase {phase} stopped at epoch {len(history)}, best epoch {best_epoch}")
    return model,history,best_epoch

def train_scenario(scenario_id,model_name,optimizer_name,learning_rate,dropout,batch_size,finetune_level=None,seed: int=42,show_epoch_log: bool=True):
    seed_everything(seed); train_loader,val_loader=build_dataloaders(batch_size,seed)
    model=build_hybrid_model(dropout) if model_name=="hybrid" else build_base_model(model_name,dropout)
    apply_finetune_level(model,finetune_level); scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
    started=time.perf_counter(); optimizer=build_optimizer(model,optimizer_name,learning_rate)
    model,h1,_=train_one_phase(model,train_loader,val_loader,optimizer,scaler,1,show_epoch_log); history=h1; epochs=[len(h1)]
    if model_name=="hybrid" and finetune_level!="frozen":
        apply_finetune_level(model,finetune_level); optimizer=build_optimizer(model,optimizer_name,learning_rate/FINE_TUNE_LR_DIVISOR)
        model,h2,_=train_one_phase(model,train_loader,val_loader,optimizer,scaler,2,show_epoch_log); history=h1+h2; epochs.append(len(h2))
    checkpoint=CHECKPOINT_DIR/f"{scenario_id}_seed{seed}.pt"; checkpoint.parent.mkdir(parents=True,exist_ok=True)
    torch.save({key:value.detach().cpu() for key,value in model.state_dict().items()},checkpoint)
    seconds=time.perf_counter()-started; del train_loader,val_loader
    return model,history,epochs,seconds

def format_metrics_table(metrics,efficiency,epochs,seconds):
    columns=["Accuracy","Precision","Recall","F1","Loss","AUC","Params","Size (MB)","Latency (ms/img)","Epochs","Train time (s)"]
    return pd.DataFrame([{**metrics,**efficiency,"Epochs":epochs,"Train time (s)":seconds}],columns=columns).round(4)

def create_confusion_heatmap(scenario_id,confusion):
    figure,axis=plt.subplots(figsize=(8,6)); image=axis.imshow(confusion,cmap="Blues"); figure.colorbar(image,ax=axis)
    axis.set(xticks=np.arange(NUM_CLASSES),yticks=np.arange(NUM_CLASSES),xticklabels=CLASS_DISPLAY_NAMES,yticklabels=CLASS_DISPLAY_NAMES,xlabel="Predicted class",ylabel="True class",title=f"Confusion Matrix - {scenario_id}")
    threshold=confusion.max()/2
    for row_index in range(NUM_CLASSES):
        for column_index in range(NUM_CLASSES): axis.text(column_index,row_index,str(confusion[row_index,column_index]),ha="center",va="center",color="white" if confusion[row_index,column_index]>threshold else "black")
    figure.tight_layout()
    return figure

def save_result(row,history,confusion):
    REPORT_DIR.mkdir(parents=True,exist_ok=True); HISTORY_DIR.mkdir(parents=True,exist_ok=True); CONFUSION_DIR.mkdir(parents=True,exist_ok=True)
    frame=pd.DataFrame([row])
    if SCREENING_PATH.exists():
        previous=pd.read_csv(SCREENING_PATH); frame=pd.concat([previous[previous.scenario_id!=row["scenario_id"]],frame],ignore_index=True)
    frame.to_csv(SCREENING_PATH,index=False); pd.DataFrame(history).to_csv(HISTORY_DIR/f"{row['scenario_id']}.csv",index=False)
    figure=create_confusion_heatmap(row["scenario_id"],confusion); figure.savefig(CONFUSION_DIR/f"{row['scenario_id']}.png",dpi=300); plt.close(figure)

def show_result_tables(scenario_id,metrics=None,efficiency=None,epochs=0,seconds=0.0,confusion=None):
    if metrics is None:
        row=pd.read_csv(SCREENING_PATH).query("scenario_id == @scenario_id").iloc[0]
        cols=["Accuracy","Precision","Recall","F1","Loss","AUC","Params","Size (MB)","Latency (ms/img)","Epochs","Train time (s)"]
        display(pd.DataFrame([row[cols].to_dict()]).round(4)); display(Image(filename=CONFUSION_DIR/f"{scenario_id}.png"))
    else:
        display(format_metrics_table(metrics,efficiency,epochs,seconds))
        figure=create_confusion_heatmap(scenario_id,confusion); display(figure); plt.close(figure)

def run_scenario(scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level=None,seed=42,rerun=False):
    if SCREENING_PATH.exists() and not rerun:
        old=pd.read_csv(SCREENING_PATH)
        if not old[(old.scenario_id==scenario_id)&(old.status=="ok")].empty: show_result_tables(scenario_id); return
    started=time.perf_counter(); instance=None; history=[]; epochs=[]; metrics={}; efficiency={}; cm=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=int); status="ok"
    try:
        instance,history,epochs,seconds=train_scenario(scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,True)
        loader=DataLoader(validation_dataset,batch_size=batch_size,shuffle=False,num_workers=0,pin_memory=True)
        y,p,prob,loss=evaluate_model(instance,loader); metrics=compute_metrics(y,p,prob,loss); efficiency=measure_efficiency(instance); cm=confusion_matrix(y,p,labels=np.arange(NUM_CLASSES)); del loader
    except torch.cuda.OutOfMemoryError:
        status="oom"; seconds=time.perf_counter()-started
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    finally:
        if instance is not None: del instance
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    row={"scenario_id":scenario_id,"model":model,"optimizer":optimizer,"lr":lr,"dropout":dropout,"batch_size":batch_size,"finetune_level":finetune_level,"seed":seed,"status":status,**{name:metrics.get(name,np.nan) for name in ("Accuracy","Precision","Recall","F1","Loss","AUC")},**{name:efficiency.get(name,np.nan) for name in ("Params","Size (MB)","Latency (ms/img)")},"epochs_per_phase":json.dumps(epochs),"Epochs":sum(epochs),"Train time (s)":seconds}
    save_result(row,history,cm); show_result_tables(scenario_id,metrics or None,efficiency,sum(epochs),seconds,cm)

def show_progress():
    results=pd.read_csv(SCREENING_PATH) if SCREENING_PATH.exists() else pd.DataFrame()
    variants=(("MobileNetV2","mobilenet",None),("EfficientNetB0","efficientnet",None),("Hybrid frozen","hybrid","frozen"),("Hybrid medium","hybrid","medium"),("Hybrid deep","hybrid","deep"))
    rows=[]
    for title,name,level in variants:
        subset=results[results.status=="ok"] if not results.empty else results
        if not subset.empty:
            subset=subset[subset.model==name]
            if level: subset=subset[subset.finetune_level==level]
        rows.append({"Model variant":title,"Scenarios done":len(subset),"Scenarios total":12})
    display(pd.DataFrame(rows))

def rank_scenarios(top_k=2):
    results=pd.read_csv(SCREENING_PATH); results=results[results.status=="ok"].sort_values(["F1","Loss"],ascending=[False,True],kind="stable")
    for name in ("mobilenet","efficientnet","hybrid"):
        ranked=results[results.model==name].head(top_k).copy(); ranked.insert(0,"Rank",np.arange(1,len(ranked)+1)); ranked.insert(1,"Model type",{"mobilenet":"MobileNetV2","efficientnet":"EfficientNetB0","hybrid":"Hybrid"}[name])
        display(ranked[["Rank","Model type","scenario_id","optimizer","lr","dropout","batch_size","finetune_level","F1","Loss","AUC","Accuracy"]].round(4))

def compare_same_configuration():
    results=pd.read_csv(SCREENING_PATH); results=results[results.status=="ok"].copy()
    results["Model variant"]=results.apply(lambda r:f"Hybrid {r.finetune_level}" if r.model=="hybrid" else ("MobileNetV2" if r.model=="mobilenet" else "EfficientNetB0"),axis=1)
    table=results.pivot_table(index=["optimizer","lr","dropout"],columns="Model variant",values="F1",aggfunc="first")
    cols=["MobileNetV2","EfficientNetB0","Hybrid frozen","Hybrid medium","Hybrid deep"]
    index=pd.MultiIndex.from_product([OPTIMIZERS,LEARNING_RATES,DROPOUT_RATES],names=["optimizer","lr","dropout"]); display(table.reindex(index=index,columns=cols).reset_index().round(4))

def run_additional_seeds(selected_scenario_ids):
    if not selected_scenario_ids: print("Select scenario IDs after reviewing the ranking."); return
    screening=pd.read_csv(SCREENING_PATH); rows=[]
    for scenario_id in selected_scenario_ids:
        cfg=screening[(screening.scenario_id==scenario_id)&(screening.status=="ok")].iloc[0]
        for seed in (43,44):
            model,history,epochs,seconds=train_scenario(scenario_id,cfg.model,cfg.optimizer,float(cfg.lr),float(cfg.dropout),int(cfg.batch_size),cfg.finetune_level if cfg.model=="hybrid" else None,seed,True)
            loader=DataLoader(validation_dataset,batch_size=int(cfg.batch_size),shuffle=False,num_workers=0,pin_memory=True)
            y,p,prob,loss=evaluate_model(model,loader); rows.append({"scenario_id":scenario_id,"seed":seed,**compute_metrics(y,p,prob,loss),**measure_efficiency(model)})
            del model,loader; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    path=REPORT_DIR/"seed_results.csv"; added=pd.DataFrame(rows)
    if path.exists():
        old=pd.read_csv(path); old=old[~old.set_index(["scenario_id","seed"]).index.isin(added.set_index(["scenario_id","seed"]).index)]; added=pd.concat([old,added],ignore_index=True)
    added.to_csv(path,index=False)
    cols=["Accuracy","Precision","Recall","F1","Loss","AUC","Params","Size (MB)","Latency (ms/img)"]
    seed42=screening[screening.scenario_id.isin(selected_scenario_ids)&(screening.status=="ok")][["scenario_id","seed",*cols]]
    merged=pd.concat([seed42,added[added.scenario_id.isin(selected_scenario_ids)][["scenario_id","seed",*cols]],],ignore_index=True).groupby("scenario_id")[cols]
    display(pd.concat([merged.mean().add_suffix(" mean"),merged.std(ddof=1).add_suffix(" std")],axis=1).round(4))

def evaluate_on_test(selected_scenario_ids):
    if not selected_scenario_ids: print("Select scenario IDs before final test evaluation."); return
    test_dataset=set_class_order(datasets.ImageFolder(DATA_ROOT/"test",transform=VALIDATION_TRANSFORM))
    rows=[]; efficiency_rows=[]; class_rows=[]; matrices={}
    for scenario_id in selected_scenario_ids:
        cfg=pd.read_csv(SCREENING_PATH).query("scenario_id == @scenario_id and status == 'ok'").iloc[0]; summed=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=int)
        for seed in (42,43,44):
            model=build_hybrid_model(float(cfg.dropout)) if cfg.model=="hybrid" else build_base_model(cfg.model,float(cfg.dropout))
            model.load_state_dict(torch.load(CHECKPOINT_DIR/f"{scenario_id}_seed{seed}.pt",map_location=DEVICE,weights_only=True))
            loader=DataLoader(test_dataset,batch_size=int(cfg.batch_size),shuffle=False,num_workers=0,pin_memory=True)
            y,p,prob,loss=evaluate_model(model,loader); metrics=compute_metrics(y,p,prob,loss); efficiency=measure_efficiency(model)
            rows.append({"scenario_id":scenario_id,"seed":seed,**metrics,**efficiency}); efficiency_rows.append({"scenario_id":scenario_id,"seed":seed,**efficiency})
            summed+=confusion_matrix(y,p,labels=np.arange(NUM_CLASSES))
            for i,name in enumerate(CLASS_DISPLAY_NAMES):
                class_rows.append({"scenario_id":scenario_id,"seed":seed,"Class":name,"Precision":precision_score(y,p,labels=[i],average="macro",zero_division=0),"Recall":recall_score(y,p,labels=[i],average="macro",zero_division=0),"F1":f1_score(y,p,labels=[i],average="macro",zero_division=0)})
            del model,loader; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        matrices[scenario_id]=pd.DataFrame(summed,index=CLASS_DISPLAY_NAMES,columns=CLASS_DISPLAY_NAMES).rename_axis("True class").rename_axis("Predicted class",axis=1); matrices[scenario_id].index=pd.MultiIndex.from_product([[f"{scenario_id} (sum of seeds 42, 43, 44)"],CLASS_DISPLAY_NAMES],names=["Scenario","True class"])
    results=pd.DataFrame(rows); results.to_csv(REPORT_DIR/"test_results.csv",index=False)
    cols=["Accuracy","Precision","Recall","F1","Loss","AUC"]; grouped=results.groupby("scenario_id")[cols]
    display(pd.concat([grouped.mean().add_suffix(" mean"),grouped.std(ddof=1).add_suffix(" std")],axis=1).round(4))
    display(pd.DataFrame(efficiency_rows).groupby("scenario_id")[["Params","Size (MB)","Latency (ms/img)"]].mean().round(4))
    for matrix in matrices.values(): display(matrix)
    display(pd.DataFrame(class_rows).groupby(["scenario_id","Class"])[["Precision","Recall","F1"]].mean().round(4))
    ranked=pd.read_csv(SCREENING_PATH); ranked=ranked[ranked.scenario_id.isin(selected_scenario_ids)&(ranked.status=="ok")].sort_values(["F1","Loss"],ascending=[False,True])
    ranked["Model variant"]=ranked.apply(lambda r:f"Hybrid {r.finetune_level}" if r.model=="hybrid" else ("MobileNetV2" if r.model=="mobilenet" else "EfficientNetB0"),axis=1)
    best=ranked.drop_duplicates("Model variant")[["scenario_id","Model variant"]]
    display(results.merge(best,on="scenario_id").groupby("Model variant")[cols].mean().T.round(4))

## D. Optuna search

In [5]:
N_TRIALS = 12
OPTUNA_STORAGE = "sqlite:///reports/optuna.db"
OPTUNA_TABLE_COLUMNS = [
    "scenario_id", "model", "optimizer", "lr", "dropout", "batch_size", "finetune_level",
    "seed", "status", "Accuracy", "Precision", "Recall", "F1", "Loss", "AUC",
    "Params", "Size (MB)", "Latency (ms/img)", "epochs_per_phase", "Epochs", "Train time (s)",
]


def create_study(model_name):
    return optuna.create_study(
        study_name=model_name,
        storage=OPTUNA_STORAGE,
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=4),
        load_if_exists=True,
    )


def run_objective(trial, model_name):
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    dropout = trial.suggest_float("dropout", 0.4, 0.5)
    finetune_level = (
        trial.suggest_categorical("finetune_level", ["frozen", "medium", "deep"])
        if model_name == "hybrid"
        else None
    )
    scenario_id = f"optuna_{model_name}_{trial.number:02d}"
    started = time.perf_counter()
    model = None
    try:
        model, history, epochs, training_seconds = train_scenario(
            scenario_id=scenario_id,
            model_name=model_name,
            optimizer_name=optimizer_name,
            learning_rate=learning_rate,
            dropout=dropout,
            batch_size=32,
            finetune_level=finetune_level,
            seed=SEED,
            show_epoch_log=False,
        )
        validation_loader = DataLoader(
            validation_dataset,
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
        )
        targets, predictions, probabilities, loss = evaluate_model(model, validation_loader)
        metrics = compute_metrics(targets, predictions, probabilities, loss)
        efficiency = measure_efficiency(model)
        trial.set_user_attr("scenario_id", scenario_id)
        trial.set_user_attr("model", model_name)
        trial.set_user_attr("finetune_level", finetune_level)
        trial.set_user_attr("metrics", metrics)
        trial.set_user_attr("efficiency", efficiency)
        trial.set_user_attr("epochs", sum(epochs))
        trial.set_user_attr("epochs_per_phase", epochs)
        trial.set_user_attr("train_time_seconds", training_seconds)
        trial.set_user_attr("status", "ok")
        return metrics["F1"]
    except torch.cuda.OutOfMemoryError:
        trial.set_user_attr("train_time_seconds", time.perf_counter() - started)
        trial.set_user_attr("status", "oom")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise
    finally:
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def print_trial_result(study, trial):
    params = dict(trial.params)
    params["batch_size"] = 32
    parameter_text = ", ".join(f"{name}={params[name]}" for name in (
        "optimizer", "lr", "dropout", "batch_size", "finetune_level"
    ) if name in params)
    metrics = trial.user_attrs.get("metrics", {})
    validation_f1 = trial.value if trial.value is not None else float("nan")
    validation_loss = metrics.get("Loss", float("nan"))
    print(
        f"trial {trial.number:02d} | model={study.study_name} | {parameter_text} | "
        f"validation F1 {validation_f1:.4f} | validation loss {validation_loss:.4f}"
    )


def trial_table_row(trial):
    metrics = trial.user_attrs.get("metrics", {})
    efficiency = trial.user_attrs.get("efficiency", {})
    return {
        "scenario_id": trial.user_attrs.get("scenario_id"),
        "model": trial.user_attrs.get("model"),
        "optimizer": trial.params.get("optimizer"),
        "lr": trial.params.get("lr"),
        "dropout": trial.params.get("dropout"),
        "batch_size": 32,
        "finetune_level": trial.user_attrs.get("finetune_level"),
        "seed": 42,
        "status": "ok" if trial.state == optuna.trial.TrialState.COMPLETE else trial.state.name.lower(),
        **metrics,
        **efficiency,
        "epochs_per_phase": json.dumps(trial.user_attrs.get("epochs_per_phase", [])),
        "Epochs": trial.user_attrs.get("epochs"),
        "Train time (s)": trial.user_attrs.get("train_time_seconds"),
    }


def run_optuna_study(model_name):
    study = create_study(model_name)
    remaining_trials = max(0, N_TRIALS - len(study.trials))
    if remaining_trials:
        study.optimize(
            lambda trial: run_objective(trial, model_name),
            n_trials=remaining_trials,
            callbacks=[print_trial_result],
            catch=(torch.cuda.OutOfMemoryError,),
            gc_after_trial=True,
            show_progress_bar=False,
        )
    completed_trials = [
        trial for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
    ]
    ranked_trials = sorted(
        completed_trials,
        key=lambda trial: (
            -trial.value,
            trial.user_attrs.get("metrics", {}).get("Loss", float("inf")),
        ),
    )
    top_rows = [trial_table_row(trial) for trial in ranked_trials[:5]]
    print("Top 5 trials")
    display(pd.DataFrame(top_rows, columns=OPTUNA_TABLE_COLUMNS).round(4))
    if ranked_trials:
        print("Best trial")
        display(pd.DataFrame([trial_table_row(ranked_trials[0])], columns=OPTUNA_TABLE_COLUMNS).round(4))
    return study

### MobileNet Study

In [6]:
mobilenet_study = run_optuna_study("mobilenet")

[I 2026-11-04 04:55:48,070] A new study created in RDB with name: mobilenet
[I 2026-11-04 04:57:20,815] Trial 0 finished with value: 0.8923723904042445 and parameters: {'optimizer': 'AdamW', 'lr': 0.000291063591313307, 'dropout': 0.4598658484197037}. Best is trial 0 with value: 0.8923723904042445.


trial 00 | model=mobilenet | optimizer=AdamW, lr=0.000291063591313307, dropout=0.4598658484197037, batch_size=32 | validation F1 0.8924 | validation loss 0.2774


[I 2026-11-04 05:04:03,435] Trial 1 finished with value: 0.9047361970843278 and parameters: {'optimizer': 'Adam', 'lr': 1.3066739238053272e-05, 'dropout': 0.48661761457749353}. Best is trial 1 with value: 0.9047361970843278.


trial 01 | model=mobilenet | optimizer=Adam, lr=1.3066739238053272e-05, dropout=0.48661761457749353, batch_size=32 | validation F1 0.9047 | validation loss 0.2654


[I 2026-11-04 05:10:44,908] Trial 2 finished with value: 0.8957002516453736 and parameters: {'optimizer': 'AdamW', 'lr': 1.0994335574766187e-05, 'dropout': 0.49699098521619944}. Best is trial 1 with value: 0.9047361970843278.


trial 02 | model=mobilenet | optimizer=AdamW, lr=1.0994335574766187e-05, dropout=0.49699098521619944, batch_size=32 | validation F1 0.8957 | validation loss 0.2915


... (17 outputs left out)

trial 11 | model=mobilenet | optimizer=Adam, lr=9.601806614992324e-05, dropout=0.46872419684486466, batch_size=32 | validation F1 0.9100 | validation loss 0.2557
Top 5 trials


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_mobilenet_09,mobilenet,Adam,0.0001,0.4713,32,None,42,ok,0.9227,...,0.9219,0.9221,0.2420,0.9888,2881796,11.2284,7.7162,[15],15,128.7906
1,optuna_mobilenet_07,mobilenet,Adam,0.0001,0.4917,32,None,42,ok,0.9185,...,0.9183,0.9181,0.2350,0.9915,2881796,11.2284,7.8009,[18],18,154.8422
2,optuna_mobilenet_08,mobilenet,Adam,0.0001,0.4986,32,None,42,ok,0.9142,...,0.9138,0.9136,0.2621,0.9885,2881796,11.2284,8.3292,[24],24,206.8902
3,optuna_mobilenet_11,mobilenet,Adam,0.0001,0.4687,32,None,42,ok,0.9099,...,0.9095,0.9100,0.2557,0.9876,2881796,11.2284,6.8483,[14],14,114.2752
4,optuna_mobilenet_03,mobilenet,Adam,0.0000,0.4183,32,None,42,ok,0.9056,...,0.9051,0.9050,0.2478,0.9888,2881796,11.2284,6.7230,[36],36,285.4471


Best trial


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_mobilenet_09,mobilenet,Adam,0.0001,0.4713,32,None,42,ok,0.9227,...,0.9219,0.9221,0.242,0.9888,2881796,11.2284,7.7162,[15],15,128.7906


### EfficientNet Study

In [7]:
efficientnet_study = run_optuna_study("efficientnet")

[I 2026-11-04 05:38:15,312] A new study created in RDB with name: efficientnet
[I 2026-11-04 05:39:57,363] Trial 0 finished with value: 0.9321344771429981 and parameters: {'optimizer': 'AdamW', 'lr': 0.000291063591313307, 'dropout': 0.4598658484197037}. Best is trial 0 with value: 0.9321344771429981.


trial 00 | model=efficientnet | optimizer=AdamW, lr=0.000291063591313307, dropout=0.4598658484197037, batch_size=32 | validation F1 0.9321 | validation loss 0.1809


[I 2026-11-04 05:47:30,621] Trial 1 finished with value: 0.9133387868421705 and parameters: {'optimizer': 'Adam', 'lr': 1.3066739238053272e-05, 'dropout': 0.48661761457749353}. Best is trial 0 with value: 0.9321344771429981.


trial 01 | model=efficientnet | optimizer=Adam, lr=1.3066739238053272e-05, dropout=0.48661761457749353, batch_size=32 | validation F1 0.9133 | validation loss 0.2584


[I 2026-11-04 05:55:24,212] Trial 2 finished with value: 0.9046948437161443 and parameters: {'optimizer': 'AdamW', 'lr': 1.0994335574766187e-05, 'dropout': 0.49699098521619944}. Best is trial 0 with value: 0.9321344771429981.


trial 02 | model=efficientnet | optimizer=AdamW, lr=1.0994335574766187e-05, dropout=0.49699098521619944, batch_size=32 | validation F1 0.9047 | validation loss 0.2728


... (17 outputs left out)

trial 11 | model=efficientnet | optimizer=Adam, lr=0.000176107753597919, dropout=0.4209575273245418, batch_size=32 | validation F1 0.9263 | validation loss 0.2230
Top 5 trials


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_efficientnet_06,efficientnet,Adam,0.0004,0.4431,32,None,42,ok,0.9356,...,0.9364,0.9358,0.1963,0.9937,4665472,18.084,10.3972,[14],14,130.4340
1,optuna_efficientnet_03,efficientnet,Adam,0.0000,0.4183,32,None,42,ok,0.9356,...,0.9352,0.9353,0.2243,0.9914,4665472,18.084,10.6818,[36],36,334.7068
2,optuna_efficientnet_00,efficientnet,AdamW,0.0003,0.4599,32,None,42,ok,0.9313,...,0.9310,0.9321,0.1809,0.9958,4665472,18.084,11.0332,[9],9,98.9127
3,optuna_efficientnet_10,efficientnet,AdamW,0.0000,0.4382,32,None,42,ok,0.9313,...,0.9310,0.9310,0.2287,0.9919,4665472,18.084,14.5279,[28],28,262.2654
4,optuna_efficientnet_09,efficientnet,AdamW,0.0001,0.4060,32,None,42,ok,0.9313,...,0.9306,0.9309,0.1960,0.9947,4665472,18.084,11.6773,[12],12,112.5009


Best trial


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_efficientnet_06,efficientnet,Adam,0.0004,0.4431,32,None,42,ok,0.9356,...,0.9364,0.9358,0.1963,0.9937,4665472,18.084,10.3972,[14],14,130.434


### Hybrid Study

In [8]:
hybrid_study = run_optuna_study("hybrid")

[I 2026-11-04 06:23:36,530] A new study created in RDB with name: hybrid
[I 2026-11-04 06:28:12,297] Trial 0 finished with value: 0.90476957612919 and parameters: {'optimizer': 'AdamW', 'lr': 0.000291063591313307, 'dropout': 0.4598658484197037, 'finetune_level': 'frozen'}. Best is trial 0 with value: 0.90476957612919.


trial 00 | model=hybrid | optimizer=AdamW, lr=0.000291063591313307, dropout=0.4598658484197037, batch_size=32, finetune_level=frozen | validation F1 0.9048 | validation loss 0.2784


[I 2026-11-04 06:33:16,545] Trial 1 finished with value: 0.9095697308730623 and parameters: {'optimizer': 'Adam', 'lr': 0.0002607024758370766, 'dropout': 0.40205844942958024, 'finetune_level': 'frozen'}. Best is trial 1 with value: 0.9095697308730623.


trial 01 | model=hybrid | optimizer=Adam, lr=0.0002607024758370766, dropout=0.40205844942958024, batch_size=32, finetune_level=frozen | validation F1 0.9096 | validation loss 0.2714


[I 2026-11-04 06:37:41,757] Trial 2 finished with value: 0.9305495918832235 and parameters: {'optimizer': 'AdamW', 'lr': 4.059611610484306e-05, 'dropout': 0.4524756431632238, 'finetune_level': 'deep'}. Best is trial 2 with value: 0.9305495918832235.


trial 02 | model=hybrid | optimizer=AdamW, lr=4.059611610484306e-05, dropout=0.4524756431632238, batch_size=32, finetune_level=deep | validation F1 0.9305 | validation loss 0.2258


... (17 outputs left out)

trial 11 | model=hybrid | optimizer=Adam, lr=0.00019743975055622854, dropout=0.4956818306289515, batch_size=32, finetune_level=deep | validation F1 0.9395 | validation loss 0.1855
Top 5 trials


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_hybrid_10,hybrid,Adam,0.0004,0.4927,32,deep,42,ok,0.9442,...,0.9439,0.9439,0.2189,0.9926,7544704,29.3144,21.0664,"[8, 10]",18,172.6854
1,optuna_hybrid_11,hybrid,Adam,0.0002,0.4957,32,deep,42,ok,0.9399,...,0.9407,0.9395,0.1855,0.9956,7544704,29.3144,19.7193,"[13, 8]",21,187.1197
2,optuna_hybrid_09,hybrid,AdamW,0.0000,0.4104,32,deep,42,ok,0.9313,...,0.9315,0.9308,0.2264,0.9907,7544704,29.3144,18.7650,"[13, 15]",28,250.7661
3,optuna_hybrid_02,hybrid,AdamW,0.0000,0.4525,32,deep,42,ok,0.9313,...,0.9312,0.9305,0.2258,0.9939,7544704,29.3144,21.2497,"[21, 8]",29,260.1547
4,optuna_hybrid_07,hybrid,AdamW,0.0000,0.4983,32,deep,42,ok,0.9227,...,0.9225,0.9225,0.1964,0.9941,7544704,29.3144,22.4154,"[16, 6]",22,200.4538


Best trial


,scenario_id,model,optimizer,lr,dropout,batch_size,finetune_level,seed,status,Accuracy,...,Recall,F1,Loss,AUC,Params,Size (MB),Latency (ms/img),epochs_per_phase,Epochs,Train time (s)
0,optuna_hybrid_10,hybrid,Adam,0.0004,0.4927,32,deep,42,ok,0.9442,...,0.9439,0.9439,0.2189,0.9926,7544704,29.3144,21.0664,"[8, 10]",18,172.6854


## E. Grid and Optuna Comparison

In [9]:
def compare_grid_and_optuna():
    if not SCREENING_PATH.exists():
        print("Grid screening results are missing.")
        return
    grid_results = pd.read_csv(SCREENING_PATH)
    comparison_rows = []
    for model_name, display_name in (
        ("mobilenet", "MobileNetV2"),
        ("efficientnet", "EfficientNetB0"),
        ("hybrid", "Hybrid"),
    ):
        model_runs = grid_results[grid_results["model"] == model_name]
        model_grid = model_runs[model_runs["status"] == "ok"]
        if not model_grid.empty:
            best_grid = model_grid.sort_values(
                ["F1", "Loss"], ascending=[False, True]
            ).iloc[0]
            grid_scenario = best_grid["scenario_id"]
            grid_f1, grid_loss, grid_auc = best_grid["F1"], best_grid["Loss"], best_grid["AUC"]
            grid_runs = len(model_runs)
            grid_seconds = model_runs["Train time (s)"].sum()
        else:
            grid_scenario = None
            grid_f1 = grid_loss = grid_auc = np.nan
            grid_runs = len(model_runs)
            grid_seconds = model_runs["Train time (s)"].sum()

        study = optuna.load_study(study_name=model_name, storage=OPTUNA_STORAGE)
        completed = [
            trial for trial in study.trials
            if trial.state == optuna.trial.TrialState.COMPLETE
        ]
        best_trial = max(
            completed,
            key=lambda trial: (
                trial.value,
                -trial.user_attrs.get("metrics", {}).get("Loss", float("inf")),
            ),
            default=None,
        )
        if best_trial is None:
            optuna_number = None
            optuna_f1 = optuna_loss = optuna_auc = np.nan
        else:
            trial_metrics = best_trial.user_attrs["metrics"]
            optuna_number = best_trial.number
            optuna_f1 = trial_metrics["F1"]
            optuna_loss = trial_metrics["Loss"]
            optuna_auc = trial_metrics["AUC"]
        optuna_runs = len(study.trials)
        optuna_seconds = sum(
            trial.user_attrs.get("train_time_seconds", 0.0)
            for trial in study.trials
        )
        comparison_rows.append({
            "Model type": display_name,
            "Grid scenario": grid_scenario,
            "Grid F1": grid_f1,
            "Grid Loss": grid_loss,
            "Grid AUC": grid_auc,
            "Grid runs": grid_runs,
            "Grid total training time (s)": grid_seconds,
            "Optuna trial": optuna_number,
            "Optuna F1": optuna_f1,
            "Optuna Loss": optuna_loss,
            "Optuna AUC": optuna_auc,
            "Optuna runs": optuna_runs,
            "Optuna total training time (s)": optuna_seconds,
        })
    display(pd.DataFrame(comparison_rows).round(4))


compare_grid_and_optuna()

,Model type,Grid scenario,Grid F1,Grid Loss,Grid AUC,Grid runs,Grid total training time (s),Optuna trial,Optuna F1,Optuna Loss,Optuna AUC,Optuna runs,Optuna total training time (s)
0,MobileNetV2,mobilenet_09,0.9269,0.2419,0.9887,12,2254.6901,9,0.9221,0.2420,0.9888,12,2512.1677
1,EfficientNetB0,efficientnet_10,0.9400,0.2156,0.9935,12,2950.3707,6,0.9358,0.1963,0.9937,12,2680.0247
2,Hybrid,hybrid_27,0.9441,0.2417,0.9948,36,9928.7028,10,0.9439,0.2189,0.9926,12,3290.5306
